In [3]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

# ============================
# 🎛️ THUMBNAIL TUNING
# ============================
RANDOM_SEED = 7
HIGHLIGHT_NAME = "Adam Ondra"

# Canvas size
FIG_WIDTH = 1728
FIG_HEIGHT = 972

# Adam placement
ADAM_X = 0.18
ADAM_Y = 60

# Named climbers you want visible
NAMED_CLIMBERS = [
    ("Will Bosi", 28),
    ("Jimmy Webb", 21),
    ("Yannick Flohé", 19),
    ("Daniel Woods", 15),
    ("Aidan Roberts", 13),
    ("Nicolai Uznik", 12),
    ("Jakob Schubert", 19),
    ("Tomoa Narasaki", 10),
    ("Noah Wheeler", 7),
    ("Jules Marchaland", 20),
    ("Vadim Timonov", 17),
]

# Manual x placements for readable thumbnail layout
MANUAL_X = {
    "Vadim Timonov": -0.44,
    "Jules Marchaland": -0.22,
    "Jakob Schubert": -0.08,
    "Noah Wheeler": -0.13,
    "Tomoa Narasaki": -0.05,
    "Nicolai Uznik": 0.12,
    "Jimmy Webb": 0.40,
    "Daniel Woods": 0.42,
    "Aidan Roberts": 0.46,
    "Yannick Flohé": 0.49,
    "Will Bosi": 0.54,
}

# Small bottom crowd
NUM_EXTRA_POINTS = 85
BOTTOM_MIN_Y = 1
BOTTOM_MAX_Y = 6
BOTTOM_ROWS = 3
BOTTOM_X_JITTER = 0.008

OUT_HTML = "adam_ondra_thumbnail_fake_data.html"
# ============================

rng = np.random.default_rng(RANDOM_SEED)

# ----------------------------
# Build forged dataset
# ----------------------------
rows = []

# Adam
rows.append({
    "Climber": HIGHLIGHT_NAME,
    "Score": ADAM_Y,
    "x": ADAM_X,
    "label": HIGHLIGHT_NAME,
    "textpos": "middle right",
    "kind": "highlight"
})

# Named climbers
for name, score in NAMED_CLIMBERS:
    rows.append({
        "Climber": name,
        "Score": score,
        "x": MANUAL_X.get(name, 0),
        "label": name,
        "textpos": "middle right" if MANUAL_X.get(name, 0) <= 0 else "middle left",
        "kind": "named"
    })

# Extra unlabeled crowd near the bottom
x_positions = np.linspace(-0.46, 0.56, NUM_EXTRA_POINTS)

for i in range(NUM_EXTRA_POINTS):
    row = i % BOTTOM_ROWS
    base_y = rng.integers(BOTTOM_MIN_Y, BOTTOM_MAX_Y + 1)
    y = base_y + row * 0.15
    x = x_positions[i] + rng.uniform(-BOTTOM_X_JITTER, BOTTOM_X_JITTER)

    rows.append({
        "Climber": f"Extra {i+1}",
        "Score": float(y),
        "x": float(x),
        "label": "",
        "textpos": "middle center",
        "kind": "crowd"
    })

rank = pd.DataFrame(rows)

# Split highlight vs others
others = rank[rank["kind"] != "highlight"].copy()
hilite = rank[rank["kind"] == "highlight"].copy()

# ----------------------------
# Plot
# ----------------------------
fig = go.Figure()

# Everyone else
fig.add_trace(go.Scatter(
    x=others["x"],
    y=others["Score"],
    mode="markers+text",
    text=others["label"],
    textposition=others["textpos"],
    textfont=dict(size=16, color="white"),
    hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
    customdata=np.stack([others["Climber"]], axis=-1),
    marker=dict(
        size=9,
        opacity=0.9,
        color="#636EFA"
    ),
    showlegend=False,
))

# Adam
fig.add_trace(go.Scatter(
    x=hilite["x"],
    y=hilite["Score"],
    mode="markers+text",
    text=hilite["label"],
    textposition="middle right",
    textfont=dict(size=22, color="white"),
    hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
    customdata=np.stack([hilite["Climber"]], axis=-1),
    marker=dict(
        size=16,
        opacity=1.0,
        symbol="diamond",
        color="tomato"
    ),
    showlegend=False,
))

# Left vertical axis line
fig.add_shape(
    type="line",
    x0=0.02, x1=0.02,
    y0=0, y1=1,
    xref="paper",
    yref="paper",
    line=dict(width=3, color="white"),
)

fig.update_layout(
    template="plotly_dark",
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
    xaxis=dict(
        title=None,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        visible=False,
        range=[-0.50, 0.58],
    ),
    yaxis=dict(
        title=None,
        showgrid=False,
        zeroline=False,
        tickfont=dict(size=18, color="white"),
        range=[0, 115],
    ),
    margin=dict(l=80, r=30, t=30, b=40),
    showlegend=False,
    plot_bgcolor="black",
    paper_bgcolor="black",
)

fig.show()
fig.write_html(OUT_HTML)
print("Saved:", OUT_HTML)

Saved: adam_ondra_thumbnail_fake_data.html
